# Take home technical exercise

## Datasets

1. Stores
2. Sales transaction

## Output

A daily summary report with the following columns

* data
* store_id
* store_name
* total_sales_nzd
* unique_customers
* average_invoice_nzd
* max_invoice_nzd
* min_inovoice_nzd

## import packages

In [1]:
import sys
from typing import List
from pathlib import Path
import pandas as pd
from pandas import DataFrame
ROOT_PATH = str(Path.cwd().resolve().parent)
if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

## Set Constants

In [2]:
STORES_DATA_PATH = r'../data/raw/stores.csv'
TRANSACTION_DATA_PATH = r'../data/raw/sales_transactions.json'
DAILY_SUMMARY_REPORT_PATH = r'../data/results/daily_sumarry.csv'

## Load the Raw Files

In [3]:
df_stores = pd.read_csv(STORES_DATA_PATH)
df_txn = pd.read_json(TRANSACTION_DATA_PATH)
df_txn['_rejection_reason'] = ""

## Perform Transformations

I realized there are serious data quality issues during the EDA.
And explicity setting the data types can help with the downstream validations.

In [4]:
#small dataset, enforce col-types
COL_TYPES = {"transaction_id":'str',	"store_id":'str',	"timestamp":'timestamp',	"customer_id":'str',	"net_amount":'float',	"store":'str'}
# enforce column types according to COL_TYPES
df_txn = df_txn.copy()
## used AI to create the below
for col, target in COL_TYPES.items():
    if col not in df_txn.columns:
        continue
    if target == "str":
        df_txn[col] = df_txn[col].astype("string")
    elif target == "float":
        df_txn[col] = pd.to_numeric(df_txn[col], errors="coerce").astype("Float64")
    elif target == "timestamp":
        df_txn[col] = pd.to_datetime(df_txn[col], utc=False,errors="coerce")
    else:
        # fallback: try pandas astype with the provided string
        try:
            df_txn[col] = df_txn[col].astype(target)
        except Exception:
            pass

df_txn.dtypes

transaction_id                          string
store_id                                string
timestamp            datetime64[us, UTC+13:00]
customer_id                             string
net_amount                             Float64
store                                   string
_rejection_reason                          str
dtype: object

## Validating Business Rules

In [5]:
valid_store_id = list(set(df_stores['store_id']))

In [ ]:
def validate_numeric_transactions(data:DataFrame, col:str)->DataFrame:
    """
    Convert the net_amount column to numeric and give reasons for rejection
    """
    df = data.copy()
    non_numeric_mask = (df['_rejection_reason'].eq("") & pd.to_numeric(df[col],errors='coerce').isna())
    df.loc[non_numeric_mask,"_rejection_reason"] = "non_numeric_amount"
    return df

def validate_store_id(data: DataFrame, stored_id: List[str], col: str) -> DataFrame:
    """
    Returns a dataframe with rejection reasons for the store id column:
    1. Missing store id
    2. Invalid store id
    """
    df = data.copy()

    missing_mask = df["_rejection_reason"].eq("") & (
        df[col].isna() | df[col].astype(str).str.strip().eq("")
    )
    invalid_mask = df["_rejection_reason"].eq("") & (~df[col].isin(stored_id)) & (~missing_mask)

    df.loc[missing_mask, "_rejection_reason"] = "missing_store"
    df.loc[invalid_mask, "_rejection_reason"] = "invalid_store_id"
    return df

def validate_transaction_id(data:DataFrame,col:str)->DataFrame:

    df = data.copy()
    empty_mask = df[col].astype(str).str.strip().eq("")
    duplicate_mask = df[col].duplicated(keep='first')
    df.loc[empty_mask,'_rejection_reason'] = "missing_transaction_id"
    df.loc[duplicate_mask,'_rejection_reason'] = "duplicate_transaction_id"
    return df

def validate_timestamp(data:DataFrame, col:str)->DataFrame:
    df = data.copy()
    timestamp_mask = (df[col].isna() & df['_rejection_reason'].eq(""))
    df.loc[timestamp_mask,'_rejection_reason'] = 'invalid_timestamp'
    return df

def run_pipeline(data:DataFrame, store_id:List[str])->DataFrame:
    df = data.copy()
    df = validate_transaction_id(data=df,col='transaction_id')
    df = validate_store_id(data=df,col='store_id',stored_id=store_id)
    df = validate_timestamp(data=df,col='timestamp')
    df = validate_numeric_transactions(data=df,col='net_amount')
    return df

def aggregate_data(transaction_data:DataFrame, sales_data:DataFrame)->DataFrame:
    transaction_data = transaction_data.copy()
    sales_data = sales_data.copy()
    #filter the valid transactions
    valid = transaction_data[transaction_data['_rejection_reason'].eq("").copy()]
    valid["date"] = pd.to_datetime(valid["timestamp"], errors="coerce").dt.date
    valid.sort_values('store_id')

    #groupby and agg
    df_agg = valid.\
    groupby(['date','store_id'])\
    .agg(
            total_sales_nzd   =("net_amount",   "sum"),
            unique_customers  =("customer_id",  "nunique"),
            average_invoice_nzd=("net_amount",  "mean"),
            max_invoice_nzd   =("net_amount",   "max"),
            min_invoice_nzd   =("net_amount",   "min"),
        ).reset_index()
    #merge on 'sales_id'
    df_fact = df_agg.merge(sales_data[['store_id','store_name']], on='store_id', how='left').sort_values(["date", "store_id"])
    df_fact = df_fact[["date", "store_id", "store_name","total_sales_nzd", "unique_customers","average_invoice_nzd", "max_invoice_nzd", "min_invoice_nzd"]]
    df_fact.to_csv(DAILY_SUMMARY_REPORT_PATH, index=False)
    return df_fact

## Run the pipeline


In [7]:
processed_df = run_pipeline(df_txn,store_id=valid_store_id)
processed_df

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,invalid_timestamp
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,missing_store
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,invalid_store_id
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,non_numeric_amount
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,missing_transaction_id


In [8]:
final_df = aggregate_data(transaction_data=processed_df,sales_data=df_stores)
final_df

,date,store_id,store_name,total_sales_nzd,unique_customers,average_invoice_nzd,max_invoice_nzd,min_invoice_nzd
0,2026-04-01,akl-001,Auckland Central,172.5,2,86.25,115.0,57.5
1,2026-04-01,wlg-001,Wellington Lambton Quay,92.0,1,92.0,92.0,92.0
2,2026-04-02,chc-001,Christchurch Riccarton,46.0,1,46.0,46.0,46.0
3,2026-04-02,wlg-001,Wellington Lambton Quay,172.5,2,86.25,138.0,34.5
